# 05 — Build historical graph features for the full dataset

Notebook 03 showed the feature idea on 100,000 rows. This notebook builds the same idea across all transactions, while keeping each transaction's features limited to earlier time steps.

We use DuckDB to read and aggregate the large CSV without loading the entire file into pandas memory. Prepared data is saved as compressed Parquet in `data/processed/`, which is excluded from Git. The first full run may take a few minutes; later runs reuse the saved Parquet files when the source data and split boundaries are unchanged.

Before running, finish notebook 04 so `data/processed/step_summary.csv` exists.

In [ ]:
import json
from pathlib import Path

import duckdb
import pandas as pd

project_root = Path.cwd()
raw_data_dir = project_root / "data" / "raw"
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
csv_files = sorted(raw_data_dir.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No PaySim CSV found in {raw_data_dir}")
data_path = csv_files[0]
step_summary_path = processed_dir / "step_summary.csv"
if not step_summary_path.exists():
    raise FileNotFoundError("Run notebooks/04_temporal_split.ipynb first to create the split summary.")
step_summary = pd.read_csv(step_summary_path, index_col="step").sort_index()
steps = step_summary.index.to_list()
train_end_step = steps[int(len(steps) * 0.70) - 1]
validation_end_step = steps[int(len(steps) * 0.85) - 1]

source_signature = {
    "file_name": data_path.name,
    "file_size_bytes": data_path.stat().st_size,
    "modified_time_ns": data_path.stat().st_mtime_ns,
}
base_path = processed_dir / "paysim_base.parquet"
base_manifest_path = processed_dir / "paysim_base_source.json"
feature_path = processed_dir / "paysim_graph_features.parquet"
feature_manifest_path = processed_dir / "paysim_graph_features_source.json"
feature_signature = {
    "source": source_signature,
    "train_end_step": int(train_end_step),
    "validation_end_step": int(validation_end_step),
}

def read_manifest(path: Path):
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return None

base_is_current = base_path.exists() and read_manifest(base_manifest_path) == source_signature
features_are_current = feature_path.exists() and read_manifest(feature_manifest_path) == feature_signature
temp_dir = (processed_dir / "duckdb_temp").resolve()
temp_dir.mkdir(parents=True, exist_ok=True)

def sql_path(path: Path) -> str:
    return path.resolve().as_posix().replace("'", "''")

connection = duckdb.connect(database=":memory:")
connection.execute("SET memory_limit = '2GB'")
connection.execute("SET threads = 2")
connection.execute(f"SET temp_directory = '{sql_path(temp_dir)}'")
print(f"Source: {data_path.name}")
print(f"Time split: train through {train_end_step}, validation through {validation_end_step}, then test")
print(f"Feature output: {feature_path}")

## 1. Convert the source CSV to a compact working Parquet file

Parquet is a compressed column format that is quicker to read than CSV. We keep only fields needed for safe model inputs, graph history, and the target label. The existing fraud flag and after-transaction balances are deliberately excluded.

In [ ]:
if base_is_current:
    print("Reusing the cached base Parquet file.")
else:
    print("Reading the CSV and writing the base Parquet file...")
    base_path.unlink(missing_ok=True)
    connection.execute(f"""
        COPY (
            SELECT
                CAST(step AS INTEGER) AS step,
                CAST(type AS VARCHAR) AS type,
                CAST(amount AS DOUBLE) AS amount,
                CAST(nameOrig AS VARCHAR) AS nameOrig,
                CAST(nameDest AS VARCHAR) AS nameDest,
                CAST(oldbalanceOrg AS DOUBLE) AS oldbalanceOrg,
                CAST(oldbalanceDest AS DOUBLE) AS oldbalanceDest,
                CAST(isFraud AS TINYINT) AS isFraud
            FROM read_csv_auto('{sql_path(data_path)}')
        ) TO '{sql_path(base_path)}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)
    base_manifest_path.write_text(json.dumps(source_signature, indent=2), encoding="utf-8")

base_rows = connection.execute(f"SELECT COUNT(*) FROM read_parquet('{sql_path(base_path)}')").fetchone()[0]
print(f"Base Parquet rows: {base_rows:,}")

## 2. Create transaction history for each account

We group transfers by account and time step, then calculate cumulative counts and amounts from **earlier steps only**. All transfers in the same step are treated as simultaneous, so a transaction cannot use another same-step transaction as past history.

In [ ]:
if features_are_current:
    print("Reusing the cached full graph-feature Parquet file.")
else:
    print("Building historical account features and writing Parquet. This is the longest step.")
    feature_path.unlink(missing_ok=True)
    connection.execute(f"""
        COPY (
            WITH account_events AS (
                SELECT
                    nameOrig AS account_id, step,
                    COUNT(*) AS outgoing_count,
                    SUM(amount) AS outgoing_amount,
                    CAST(0 AS BIGINT) AS incoming_count,
                    CAST(0 AS DOUBLE) AS incoming_amount
                FROM read_parquet('{sql_path(base_path)}')
                GROUP BY nameOrig, step
                UNION ALL
                SELECT
                    nameDest AS account_id, step,
                    CAST(0 AS BIGINT) AS outgoing_count,
                    CAST(0 AS DOUBLE) AS outgoing_amount,
                    COUNT(*) AS incoming_count,
                    SUM(amount) AS incoming_amount
                FROM read_parquet('{sql_path(base_path)}')
                GROUP BY nameDest, step
            ),
            account_step_activity AS (
                SELECT
                    account_id, step,
                    SUM(outgoing_count) AS outgoing_count,
                    SUM(outgoing_amount) AS outgoing_amount,
                    SUM(incoming_count) AS incoming_count,
                    SUM(incoming_amount) AS incoming_amount
                FROM account_events
                GROUP BY account_id, step
            ),
            historical_activity AS (
                SELECT
                    account_id, step,
                    COALESCE(SUM(outgoing_count) OVER (
                        PARTITION BY account_id ORDER BY step
                        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
                    ), 0) AS prior_outgoing_count,
                    COALESCE(SUM(outgoing_amount) OVER (
                        PARTITION BY account_id ORDER BY step
                        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
                    ), 0) AS prior_outgoing_amount,
                    COALESCE(SUM(incoming_count) OVER (
                        PARTITION BY account_id ORDER BY step
                        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
                    ), 0) AS prior_incoming_count,
                    COALESCE(SUM(incoming_amount) OVER (
                        PARTITION BY account_id ORDER BY step
                        ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
                    ), 0) AS prior_incoming_amount
                FROM account_step_activity
            )
            SELECT
                tx.step, tx.type, tx.amount,
                tx.oldbalanceOrg, tx.oldbalanceDest,
                COALESCE(sender.prior_outgoing_count, 0) AS prior_outgoing_count,
                COALESCE(sender.prior_outgoing_amount, 0) AS prior_outgoing_amount,
                COALESCE(receiver.prior_incoming_count, 0) AS prior_incoming_count,
                COALESCE(receiver.prior_incoming_amount, 0) AS prior_incoming_amount,
                tx.nameOrig, tx.nameDest, tx.isFraud,
                CASE
                    WHEN tx.step <= {int(train_end_step)} THEN 'train'
                    WHEN tx.step <= {int(validation_end_step)} THEN 'validation'
                    ELSE 'test'
                END AS data_split
            FROM read_parquet('{sql_path(base_path)}') AS tx
            LEFT JOIN historical_activity AS sender
                ON sender.account_id = tx.nameOrig AND sender.step = tx.step
            LEFT JOIN historical_activity AS receiver
                ON receiver.account_id = tx.nameDest AND receiver.step = tx.step
        ) TO '{sql_path(feature_path)}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)
    feature_manifest_path.write_text(json.dumps(feature_signature, indent=2), encoding="utf-8")

print(f"Saved full graph features to: {feature_path}")

## 3. Check the prepared output

The output keeps account IDs for case investigation, but they are not included in `model_features`. `isFraud` is the target label. The `data_split` column records the chronological period for each transaction.

In [ ]:
feature_file = sql_path(feature_path)
feature_summary = connection.execute(f"""
    SELECT
        data_split, MIN(step) AS first_step, MAX(step) AS last_step,
        COUNT(*) AS transactions, SUM(isFraud) AS fraud_transactions,
        100.0 * SUM(isFraud) / COUNT(*) AS fraud_rate_percent
    FROM read_parquet('{feature_file}')
    GROUP BY data_split
    ORDER BY CASE data_split WHEN 'train' THEN 1 WHEN 'validation' THEN 2 ELSE 3 END
""").df()
display(feature_summary)

feature_checks = connection.execute(f"""
    SELECT
        COUNT(*) AS feature_rows,
        SUM(isFraud) AS fraud_labels,
        COUNT(*) FILTER (WHERE prior_outgoing_count IS NULL OR prior_incoming_count IS NULL) AS missing_history_values,
        COUNT(*) FILTER (WHERE prior_outgoing_count < 0 OR prior_incoming_count < 0) AS negative_history_counts
    FROM read_parquet('{feature_file}')
""").df()
display(feature_checks)

example_rows = connection.execute(f"""
    SELECT step, type, amount, prior_outgoing_count, prior_incoming_count, isFraud, data_split
    FROM read_parquet('{feature_file}')
    WHERE prior_outgoing_count > 0 OR prior_incoming_count > 0
    LIMIT 10
""").df()
display(example_rows)
expected_rows = int(step_summary["transactions"].sum())
actual_rows = int(feature_checks.loc[0, "feature_rows"])
assert actual_rows == expected_rows, f"Expected {expected_rows:,} rows but found {actual_rows:,}"
assert int(feature_checks.loc[0, "missing_history_values"]) == 0
assert int(feature_checks.loc[0, "negative_history_counts"]) == 0
print("Full-data row-count and graph-history checks passed.")

## Why these features are graph features

A sender's prior outgoing count/amount describes earlier edges leaving that account. A receiver's prior incoming count/amount describes earlier edges entering it. These are simple directed-network activity features. They are safer than using a graph built from all transactions at once, which would let future transfers influence past predictions.

This is the full-data feature layer, not the final model. The next stage will train models on the `train` rows, choose settings on `validation`, and report final performance on `test`.